# Notebook 6: GPU, CUDA & Mixed Precision (OPTIONAL)

> **This notebook is OPTIONAL.**  Everything runs safely on a CPU-only machine.  GPU-specific code is guarded with `if torch.cuda.is_available()` and wrapped in `try/except`.

---

## Learning Objectives

By the end of this notebook you will be able to:

- Detect whether a GPU is available and select a device
- Move tensors and models between CPU and GPU using `.to(device)` and `.cuda()`
- Use `pin_memory` and `non_blocking` for faster host-to-device transfer
- Apply mixed precision training with `torch.cuda.amp` (`autocast` + `GradScaler`)
- Benchmark CPU vs GPU training speed (when a GPU is available)

## Prerequisites

- Notebooks 01-05 of this series
- Basic understanding of training loops (Notebook 05)
- PyTorch installed (GPU build optional)

## Table of Contents

1. [Device Detection](#1)
2. [Moving Data & Models to GPU](#2)
3. [`pin_memory` and `non_blocking`](#3)
4. [CPU vs GPU Benchmark](#4)
5. [Mixed Precision Training](#5)
6. [Full GPU-Enabled Training Loop](#6)
7. [Common Mistakes & Debugging Tips](#7)
8. [Exercises](#8)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version:    {torch.version.cuda}")
    print(f"GPU name:        {torch.cuda.get_device_name(0)}")
    print(f"GPU count:       {torch.cuda.device_count()}")

<a id='1'></a>
## 1. Device Detection

The standard PyTorch idiom for device-agnostic code:

In [ ]:
# The canonical device-selection pattern
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# For Apple Silicon Macs (MPS backend)
# Uncomment if on macOS with M1/M2/M3:
# if torch.backends.mps.is_available():
#     device = torch.device("mps")

In [ ]:
# Additional GPU info (safe on CPU)
try:
    if torch.cuda.is_available():
        print(f"Current device index: {torch.cuda.current_device()}")
        print(f"Device name:          {torch.cuda.get_device_name(0)}")
        mem = torch.cuda.get_device_properties(0).total_mem
        print(f"Total GPU memory:     {mem / 1e9:.2f} GB")
    else:
        print("No CUDA device found. Running on CPU.")
        print("(This is fine -- all code in this notebook works on CPU too.)")
except Exception as e:
    print(f"Error querying GPU: {e}")

<a id='2'></a>
## 2. Moving Data & Models to GPU

There are two equivalent ways:

| Method | Works on | Notes |
|---|---|---|
| `.to(device)` | Any device string/object | **Recommended** -- device-agnostic |
| `.cuda()` | CUDA GPUs only | Shorthand, hardcodes CUDA |

**Important:** For tensors, `.to()` returns a **new** tensor (unless already on the target device).  For `nn.Module`, `.to()` moves the model **in-place** and returns `self`.

In [ ]:
# Moving tensors
x_cpu = torch.randn(3, 3)
print(f"x_cpu device: {x_cpu.device}")

x_device = x_cpu.to(device)
print(f"x_device device: {x_device.device}")

# Moving back to CPU (needed for NumPy, matplotlib, etc.)
x_back = x_device.cpu()
print(f"x_back device: {x_back.device}")

In [ ]:
# Moving a model
class SimpleNet(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)

model = SimpleNet(10, 64, 3).to(device)   # move all parameters to device

# Verify parameter devices
for name, param in model.named_parameters():
    print(f"{name:20s} -> {param.device}")

In [ ]:
# Common pattern: create data on device directly
x_on_device = torch.randn(5, 10, device=device)
print(f"Created directly on {x_on_device.device}")

# Forward pass -- input and model must be on the same device
output = model(x_on_device)
print(f"Output device: {output.device}, shape: {output.shape}")

<a id='3'></a>
## 3. `pin_memory` and `non_blocking`

When using a `DataLoader` with GPU training, two settings speed up CPU-to-GPU transfers:

- **`pin_memory=True`** in `DataLoader`: allocates data in page-locked (pinned) memory, enabling faster DMA copies to GPU
- **`non_blocking=True`** in `.to(device)`: allows the transfer to overlap with other operations (asynchronous)

```python
# Typical usage:
loader = DataLoader(dataset, batch_size=64, pin_memory=True, num_workers=4)

for X_batch, y_batch in loader:
    X_batch = X_batch.to(device, non_blocking=True)
    y_batch = y_batch.to(device, non_blocking=True)
    ...
```

**When to use:**
- `pin_memory=True`: always when training on GPU with `DataLoader`
- `non_blocking=True`: when you want to overlap data transfer with computation
- On CPU-only machines these flags are silently ignored (safe to include)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Create a simple dataset
torch.manual_seed(42)
X_demo = torch.randn(1000, 10)
y_demo = torch.randint(0, 3, (1000,))

dataset = TensorDataset(X_demo, y_demo)

# pin_memory only helps with CUDA; harmless on CPU
use_pin = torch.cuda.is_available()
loader = DataLoader(dataset, batch_size=64, shuffle=True,
                    pin_memory=use_pin, num_workers=0)

# Iterate one batch to verify
for X_b, y_b in loader:
    X_b = X_b.to(device, non_blocking=True)
    y_b = y_b.to(device, non_blocking=True)
    print(f"Batch X on {X_b.device}, y on {y_b.device}")
    break

<a id='4'></a>
## 4. CPU vs GPU Benchmark

Let us time a training loop on CPU and GPU (if available).  If no GPU is present, expected results are shown.

In [ ]:
def benchmark_training(device_str, input_dim=784, hidden_dim=512,
                       output_dim=10, batch_size=256, n_batches=100):
    """Time `n_batches` forward+backward passes on the given device."""
    dev = torch.device(device_str)
    torch.manual_seed(42)

    model = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim),
    ).to(dev)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # Warm-up
    x = torch.randn(batch_size, input_dim, device=dev)
    y = torch.randint(0, output_dim, (batch_size,), device=dev)
    _ = model(x)
    if dev.type == "cuda":
        torch.cuda.synchronize()

    start = time.time()
    for _ in range(n_batches):
        x = torch.randn(batch_size, input_dim, device=dev)
        y = torch.randint(0, output_dim, (batch_size,), device=dev)

        logits = model(x)
        loss = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if dev.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.time() - start
    return elapsed


# Run benchmark
try:
    cpu_time = benchmark_training("cpu")
    print(f"CPU:  {cpu_time:.3f}s for 100 batches")

    if torch.cuda.is_available():
        gpu_time = benchmark_training("cuda")
        print(f"GPU:  {gpu_time:.3f}s for 100 batches")
        print(f"Speedup: {cpu_time / gpu_time:.1f}x")
    else:
        print("\n(No GPU detected. Typical speedups range from 5x to 50x")
        print(" depending on model size, batch size, and GPU model.)")
except Exception as e:
    print(f"Benchmark error: {e}")

<a id='5'></a>
## 5. Mixed Precision Training

Mixed precision uses **float16** (half precision) for most operations and **float32** for numerically sensitive parts (loss scaling, weight updates).

**Benefits:**
- ~2x memory reduction
- ~2-3x faster on GPUs with Tensor Cores (Volta, Turing, Ampere, Hopper)

**Two key components:**

| Component | Purpose |
|---|---|
| `torch.cuda.amp.autocast` | Automatically casts operations to float16 where safe |
| `torch.cuda.amp.GradScaler` | Scales the loss to prevent float16 underflow in gradients |

The gradient scaling works as follows:

$$\mathcal{L}_{\text{scaled}} = s \cdot \mathcal{L}, \quad \nabla w = \frac{1}{s} \nabla_w \mathcal{L}_{\text{scaled}}$$

where $s$ is a dynamic scale factor managed by `GradScaler`.

In [ ]:
# Mixed precision training example
# This runs on CPU too (autocast is a no-op on CPU, GradScaler is guarded)

try:
    torch.manual_seed(42)

    model_amp = nn.Sequential(
        nn.Linear(784, 256),
        nn.ReLU(),
        nn.Linear(256, 10),
    ).to(device)

    loss_fn_amp = nn.CrossEntropyLoss()
    optimizer_amp = optim.Adam(model_amp.parameters(), lr=1e-3)

    # GradScaler only works on CUDA
    use_amp = torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    # Simulated data
    X_amp = torch.randn(256, 784, device=device)
    y_amp = torch.randint(0, 10, (256,), device=device)

    for step in range(5):
        optimizer_amp.zero_grad()

        # autocast: operations inside run in float16 (on CUDA) or float32 (on CPU)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model_amp(X_amp)
            loss = loss_fn_amp(logits, y_amp)

        # Scaled backward
        scaler.scale(loss).backward()

        # Unscale gradients, then step
        scaler.step(optimizer_amp)
        scaler.update()

        print(f"Step {step}: loss = {loss.item():.4f}")

    print(f"\nMixed precision enabled: {use_amp}")
    print("Training completed successfully.")

except Exception as e:
    print(f"Mixed precision example failed (expected on some CPU setups): {e}")
    print("This is fine -- mixed precision is a GPU optimization.")

<a id='6'></a>
## 6. Full GPU-Enabled Training Loop

Putting it all together: a complete, device-agnostic training loop with optional mixed precision.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

# ---- Config ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()
BATCH_SIZE = 64
NUM_EPOCHS = 20
LR = 1e-3

print(f"Device: {device}")
print(f"AMP enabled: {use_amp}")

# ---- Synthetic dataset ----
n_samples = 2000
n_features = 20
n_classes = 5

X_full = torch.randn(n_samples, n_features)
y_full = torch.randint(0, n_classes, (n_samples,))

# Train/Val split
n_train = int(0.8 * n_samples)
X_train, X_val = X_full[:n_train], X_full[n_train:]
y_train, y_val = y_full[:n_train], y_full[n_train:]

train_ds = TensorDataset(X_train, y_train)
val_ds   = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=use_amp, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          pin_memory=use_amp, num_workers=0)

# ---- Model ----
model_gpu = nn.Sequential(
    nn.Linear(n_features, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, n_classes),
).to(device)

loss_fn_gpu = nn.CrossEntropyLoss()
optimizer_gpu = optim.Adam(model_gpu.parameters(), lr=LR)
scaler_gpu = torch.cuda.amp.GradScaler(enabled=use_amp)

# ---- Training loop ----
try:
    for epoch in range(NUM_EPOCHS):
        # ---- Train ----
        model_gpu.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for X_b, y_b in train_loader:
            X_b = X_b.to(device, non_blocking=True)
            y_b = y_b.to(device, non_blocking=True)

            optimizer_gpu.zero_grad()

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model_gpu(X_b)
                loss = loss_fn_gpu(logits, y_b)

            scaler_gpu.scale(loss).backward()
            scaler_gpu.step(optimizer_gpu)
            scaler_gpu.update()

            train_loss += loss.item() * X_b.size(0)
            train_correct += (logits.argmax(1) == y_b).sum().item()
            train_total += X_b.size(0)

        # ---- Validate ----
        model_gpu.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b = X_b.to(device, non_blocking=True)
                y_b = y_b.to(device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=use_amp):
                    logits = model_gpu(X_b)
                    loss = loss_fn_gpu(logits, y_b)

                val_loss += loss.item() * X_b.size(0)
                val_correct += (logits.argmax(1) == y_b).sum().item()
                val_total += X_b.size(0)

        # ---- Log ----
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
                  f"Train Loss: {train_loss/train_total:.4f}  "
                  f"Acc: {train_correct/train_total:.4f} | "
                  f"Val Loss: {val_loss/val_total:.4f}  "
                  f"Acc: {val_correct/val_total:.4f}")

    print("\nTraining complete.")

except Exception as e:
    print(f"Training error: {e}")
    print("If on CPU, try setting use_amp = False.")

<a id='7'></a>
## 7. Common Mistakes & Debugging Tips

| Mistake | Error message / symptom | Fix |
|---|---|---|
| Input on CPU, model on GPU | `RuntimeError: expected all tensors to be on the same device` | Move input with `.to(device)` |
| Calling `.numpy()` on a CUDA tensor | `Can't call numpy() on Tensor that requires grad...` | Use `.detach().cpu().numpy()` |
| Forgetting `torch.cuda.synchronize()` in timing | Incorrect (too fast) GPU timing | Add `synchronize()` before start/end |
| Using `pin_memory` without CUDA | No error, but no benefit either | Only set `True` when GPU is available |
| Not using `scaler.update()` with AMP | Scale factor never adjusts | Always call `scaler.update()` after `scaler.step()` |
| `GradScaler` on CPU | May raise error in some versions | Guard with `enabled=torch.cuda.is_available()` |
| GPU out of memory | `CUDA out of memory` | Reduce batch size, use gradient accumulation, or use AMP |

### GPU Memory Management

```python
# Check GPU memory usage
if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
    print(f"Cached:    {torch.cuda.memory_reserved() / 1e6:.1f} MB")

    # Free cached memory
    torch.cuda.empty_cache()
```

In [ ]:
# Safe GPU memory check
try:
    if torch.cuda.is_available():
        print(f"Allocated: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
        print(f"Cached:    {torch.cuda.memory_reserved() / 1e6:.1f} MB")
        torch.cuda.empty_cache()
        print("Cache cleared.")
    else:
        print("No CUDA device -- memory stats not available.")
except Exception as e:
    print(f"Memory check error: {e}")

<a id='8'></a>
## 8. Exercises

### Exercise 1: Convert a CPU Training Loop to GPU-Enabled

The code below trains a model entirely on CPU.  Modify it so that:

1. It auto-detects the device (CPU or CUDA)
2. The model and data are moved to the correct device
3. (Bonus) Add mixed precision support with `autocast` + `GradScaler`

```python
# ---- CPU-only code to convert ----
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(50, 128),
    nn.ReLU(),
    nn.Linear(128, 5),
)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

X = torch.randn(500, 50)
y = torch.randint(0, 5, (500,))

for epoch in range(50):
    model.train()
    logits = model(X)
    loss = loss_fn(logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        acc = (logits.argmax(1) == y).float().mean()
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Acc: {acc.item():.4f}")
```

In [ ]:
# YOUR CODE HERE

### Exercise 1 -- Solution

<details>
<summary>Click to expand</summary>

In [ ]:
# ---- Solution: GPU-enabled training loop ----
try:
    torch.manual_seed(42)

    # 1. Device detection
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()
    print(f"Device: {device}, AMP: {use_amp}")

    # 2. Model on device
    model = nn.Sequential(
        nn.Linear(50, 128),
        nn.ReLU(),
        nn.Linear(128, 5),
    ).to(device)                                   # <-- moved

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)  # <-- AMP

    # 3. Data on device
    X = torch.randn(500, 50, device=device)        # <-- on device
    y = torch.randint(0, 5, (500,), device=device) # <-- on device

    for epoch in range(50):
        model.train()
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):   # <-- AMP
            logits = model(X)
            loss = loss_fn(logits, y)

        scaler.scale(loss).backward()              # <-- scaled backward
        scaler.step(optimizer)                     # <-- scaled step
        scaler.update()                            # <-- update scale

        if (epoch + 1) % 10 == 0:
            with torch.no_grad():
                acc = (logits.argmax(1) == y).float().mean()
            print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Acc: {acc.item():.4f}")

    print("Done!")

except Exception as e:
    print(f"Error: {e}")

</details>

### Exercise 2: Memory Profiling

If you have a GPU, write code that:
1. Creates a large tensor (e.g., 10000 x 10000) on GPU
2. Prints the allocated memory before and after
3. Deletes the tensor and calls `torch.cuda.empty_cache()`
4. Prints the allocated memory again

If no GPU is available, write the code with appropriate guards and explain what the expected output would be.

<details>
<summary>Click for solution</summary>

```python
if torch.cuda.is_available():
    print(f"Before: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
    
    big = torch.randn(10000, 10000, device='cuda')
    print(f"After alloc: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
    # Expected: ~400 MB (10000*10000*4 bytes)
    
    del big
    torch.cuda.empty_cache()
    print(f"After free: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
else:
    print("No GPU available.")
    print("Expected output: Before ~0 MB, After alloc ~400 MB, After free ~0 MB")
```
</details>

---

*End of Notebook 6 (OPTIONAL).*